# Zero-Shot Topic Classification with BERTopic

Assigns each Korean K-pop article to one of four predefined categories using sentence-BERT embeddings,
with BERTopic's clustering pipeline disabled.

In [3]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from konlpy.tag import Okt
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.cluster import BaseCluster

In [4]:
BASE_DIR = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/article")
INPUT_CSV = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/top300_filtered.csv")
OUTPUT_CSV = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/top300_filtered_with_topics_ZeroShot.csv")
ARTICLES_DIR = BASE_DIR

**Preprocessing.** Read each article and extract Korean nouns (≥2 chars) with Okt. Tokenized text gives
cleaner c-TF-IDF keywords than raw text.

In [5]:
def load_and_preprocess_documents(csv_path, articles_dir):
    """
    Loads metadata, reads article files, and tokenizes Korean text.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found at: {csv_path}")

    df = pd.read_csv(csv_path)
    okt = Okt()
    processed_docs = []
    valid_indices = []

    print("Processing documents...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        file_name = Path(row['file_path']).name
        full_path = articles_dir / file_name

        try:
            with open(full_path, 'r', encoding='utf-8') as f:
                text = f.read()
                
            # Tokenize using KoNLPy (Okt)
            tokens = okt.nouns(text)
            tokens = [word for word in tokens if len(word) > 1]
            processed_text = " ".join(tokens)
            
            processed_docs.append(processed_text)
            valid_indices.append(idx)
            
        except Exception as e:
            print(f"Warning: Could not read file {file_name}. Error: {e}")

    df_clean = df.loc[valid_indices].copy()
    return df_clean, processed_docs


**Zero-shot model.** Each article is assigned to whichever predefined label has the closest cosine
similarity to its embedding. `BaseDimensionalityReduction` and `BaseCluster` disable UMAP/HDBSCAN so no
new clusters are discovered. `zeroshot_min_similarity=0.01` forces every article into a label.

In [ ]:
# Zero-Shot BERTopic Function
def run_topic_modeling():
    df, documents = load_and_preprocess_documents(INPUT_CSV, ARTICLES_DIR)
    documents = [str(doc) for doc in documents]

    zeroshot_topic_list = [
        "스타", 
        "결혼 연애", 
        "사망", 
        "논란"
    ]

    # Manually set stop words
    korean_stop_words = ['통해', '단독', '♥', '라며', '사진', '기자', '방송', '뉴스', '텐아시아', '배우', '사람', '스포츠조선', '엑스포츠', 'OSEN', '소식', '스타뉴스', '마이데일리']
    
    vectorizer_model = CountVectorizer(stop_words=korean_stop_words)
    embedding_model = SentenceTransformer("jhgan/ko-sbert-sts")
    
    print("Encoding documents...")
    embeddings = embedding_model.encode(documents, show_progress_bar=True, batch_size=4)

    empty_dimensionality_model = BaseDimensionalityReduction()
    empty_cluster_model = BaseCluster()

    print("Fitting pure Zero-Shot BERTopic model...")
    topic_model = BERTopic(
        umap_model=empty_dimensionality_model,
        hdbscan_model=empty_cluster_model,
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        zeroshot_topic_list=zeroshot_topic_list,   
        zeroshot_min_similarity=0.01, # Keep low to force assignment
    )
    
    topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)

    df['topic'] = topics
    
    unique_topics = set(topics)
    num_topics = len(unique_topics) - 1 if -1 in unique_topics else len(unique_topics)
    print(f"Success: Found {num_topics} topics.")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Results saved: {OUTPUT_CSV}")

    return topic_model, df

**Run.** Topic IDs aren't guaranteed to match the order of `zeroshot_topic_list` — read the `Name`
column to see what each cluster actually is.

In [7]:
topic_model, final_df = run_topic_modeling()

display(topic_model.get_topic_info())

Processing documents...


 50%|████▉     | 149/300 [00:06<00:06, 25.15it/s]

 94%|█████████▍| 282/300 [00:10<00:00, 28.23it/s]

100%|██████████| 300/300 [00:11<00:00, 26.31it/s]


Encoding documents...


Batches: 100%|██████████| 75/75 [00:34<00:00,  2.14it/s]


Fitting pure Zero-Shot BERTopic model...
Success: Found 4 topics.
Results saved: C:\Users\WINDOWS 11\Desktop\kpop_agenda\data\top300_filtered_with_topics_ZeroShot.csv


,Topic,Count,Name,Representation,Representative_Docs
0,0,150,0_박나래_매니저_공개_주장,"[박나래, 매니저, 공개, 주장, 논란, 모습, 영상, 사실, 자신, 대해]",[스타 뉴스 최혜진 기자 코미디언 박나래 사진 이동훈 코미디언 박나래 대한 자극 사...
1,1,64,1_공개_우리_모습_아시아,"[공개, 우리, 모습, 아시아, 멤버, 유튜브, 출연, 웃음, 조세호, 미니시리즈]",[스포츠조선 조윤선 기자 가수 서태지 크리스마스 근황 서태지 성탄절 이브 라며 장문...
2,2,56,2_결혼_결혼식_공개_부부,"[결혼, 결혼식, 공개, 부부, 아내, 이혼, 최준희, 지난, 모습, 유튜브]",[김종국 마이데일리 김진 기자 머리칼 하나 공개 지난해 결혼 김종국 꽁꽁 아내 정체...
3,3,28,3_사망_지난_선수_활동,"[사망, 지난, 선수, 활동, 김혜수, 향년, 세상, 원혁, 영화, 치료]",[우리 방금 결혼 스틸컷 스포츠조선 기자 할리우드 배우 브리트니 머피 사망 사망 경...


**Top 5 by views per topic.** 

In [8]:
TOP5_CSV_PATH = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/top5_articles_per_topic_ZeroShot.csv")

def save_top_5_by_views(df, output_path):
    print("Extracting the top 5 most viewed articles per topic...")
    
    valid_topics = df[df['topic'] != -1].copy()

    # Safeguard: Ensure 'views' is a number
    valid_topics['views'] = pd.to_numeric(valid_topics['views'], errors='coerce')

    top5_df = (valid_topics
               .sort_values(by=['topic', 'views'], ascending=[True, False])
               .groupby('topic')
               .head(5))

    top5_df.to_csv(output_path, index=False)
    print(f"Top 5 most viewed articles per topic saved to: {output_path}")
    
    # Preview
    display(top5_df[['topic', 'views', 'title', 'file_path']])

save_top_5_by_views(final_df, TOP5_CSV_PATH)

Extracting the top 5 most viewed articles per topic...
Top 5 most viewed articles per topic saved to: C:\Users\WINDOWS 11\Desktop\kpop_agenda\data\top5_articles_per_topic_ZeroShot.csv


,topic,views,title,file_path
0,0,1823020,"신기루, 법정 구속됐다…""혐의 인정 못해"" 결국 상습 허언죄로 입소 ('배불리힐스')",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
1,0,1809646,"박나래 논란, 새 폭로로 ‘국면 전환’… 전 매니저 “잠든 박나래에 주사이모 계속 ...",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
2,0,1586302,"이진호 ""박나래 매니저들, 55억 이태원 자택 도둑 사건後 큰 배신감…폭로 촉발 결...",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
3,0,1527854,"""성관계 있었지만 억지로 한 것 아니다"" 소속 배우 '성폭행 혐의' 체포된 기획사 대표",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
7,0,1449491,"“소리 지르며 촬영장 이탈”.. 박나래, 논란 속 ‘놀토’서 돌발 해프닝 ('놀토')",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
4,1,1484542,"유재석, 결국 비난 쏟아냈다…""정말 최악, 줘도 안 가져"" 허접한 비밀 선물에 격분...",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
8,1,1425348,"'한고은♥' 신영수, 수년째 무직…""스카우트 無, 백수란 말 안 괜찮아"" [마데핫리뷰]",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
11,1,1404859,"'윤상현♥' 메이비, 갑작스러운 비보에 자책 ""널 살릴 수 있었던 시간은 언제였을까""",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
13,1,1374750,"김대호, 제대로 사고쳤다…달동네 2억집 싹 고쳤는데 ""20박스 모래 부어"" ('나혼산')",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
17,1,1343691,"""항의 빗발쳐""…김수용 영상, 결국 삭제됐다 ""한시간 만에 내려"" ('라스')",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
